### Testing knn accuracy of TF-IDF models

In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import datasets
import src
from src.knn_accuracy import knn_accuracy
from mteb.evaluation.evaluators.utils import get_vocab

In [2]:
# load datasets
data_batched = {
"arxiv": datasets.load_dataset("mteb/arxiv-clustering-p2p", revision="a122ad7f3f0291bf49cc6f4d32aa80929df69d5d")["test"],
"biorxiv": datasets.load_dataset("mteb/biorxiv-clustering-p2p", revision="f5dbc242e11dd8e24def4c4268607a49e02946dc")["test"],
"medrxiv": datasets.load_dataset("mteb/medrxiv-clustering-p2p", revision="e7a26af6f3ae46b30dde8737f02c07b1505bcc73")["test"],
"reddit": datasets.load_dataset("mteb/reddit-clustering-p2p", revision="385e3cb46b4cfa89021f56c4380204149d0efe33")["test"],
"stackexchange": datasets.load_dataset("mteb/stackexchange-clustering-p2p", revision="815ca46b2622cec33ccafc3735d572c266efdb44")["test"]
}

In [3]:
data_full = {}
for name, data in data_batched.items():
    if name == "biorxiv":
        labels = [split["labels"] for split in data]
        sentences = [split["sentences"] for split in data]
    else:    
        labels = [x for split in data for x in split["labels"]]
        sentences = [x for split in data for x in split["sentences"]]
    data_full[name] = {"sentences": sentences, "labels": labels}

dataframe
colums: models
rows: datasets (full and batched)

Models to test:
- tfidf_log 
- tfidf_svd_log
- tfidf_svd_log_piecewise

#### logarithmic TF-IDF

In [ ]:
model = src.tfidf_log.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)
    if name in ["arxiv", "reddit"]:
        # arxiv and reddit are too big to perform clustering on full tfidf representations
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    if name == "biorxiv": # biorxiv is the only one without batches
        continue
    # get full data to compute unified vocabulary
    vocab = get_vocab(data_full[name]["sentences"])
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 168265)
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 73110)
reddit
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 120133)
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)


#### logarithmic TF-IDF with svd reduction

In [ ]:
def knn_acc_svd(model, dataset = data_full, dataset_batched = data_batched):

    scores = {}
    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)
        # compute unified vocabulary (only necessary because the svd models need a vocab input)
        vocab = get_vocab(data["sentences"])
        # compute svd components
        v = model.encode(sentences=data["sentences"], vocab = vocab)
        # compute svd reduced embeddings
        embeddings = model.encode(sentences=data["sentences"], vocab = vocab, V = v)
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        
        if name == "biorxiv": # biorxiv is the only one without batches
            continue
        # get full data to compute unified vocabulary
        vocab = get_vocab(dataset[name]["sentences"])
        # compute svd components
        v = model.encode(sentences=dataset[name]["sentences"], vocab = vocab)
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab, V = v)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
        scores[name + "_batchwise"] = batch_scores
    

    return scores


In [5]:
model = src.tfidf_svd_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"../MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(732723, 331735)
V not provided, fit SVD to get V
tfidf matrix shape(732723, 331735)
V available, perform SVD dim reduction
dense matrix shape(732723, 100)


/opt/anaconda3/envs/hertie/lib/python3.12/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


biorxiv
get_vocab called!
input: <class 'list'> of length 53787
vocab of lenght 164697 starting with ['0001x', '000bp', '000km', '000s', '000x', '001x', '0020t', '0025mg', '003g', '003ml']
tfidf matrix shape(53787, 164697)
V not provided, fit SVD to get V
tfidf matrix shape(53787, 164697)
V available, perform SVD dim reduction
dense matrix shape(53787, 100)
medrxiv
get_vocab called!
input: <class 'list'> of length 37500
vocab of lenght 68780 starting with ['0001for', '000bdt', '000copies', '000gbp', '000th', '000x', '001and', '001no', '002ng', '004x']
tfidf matrix shape(37500, 68780)
V not provided, fit SVD to get V
tfidf matrix shape(37500, 68780)
V available, perform SVD dim reduction
dense matrix shape(37500, 100)
reddit
get_vocab called!
input: <class 'list'> of length 459399
vocab of lenght 307965 starting with ['00000000004a0048', '00000000004c004a', '000000000143aaa0', '00000000030a8678', '00000000030a8730', '00000000030a8c40', '00000000030a8da8', '000000002usd', '00000001s', '0

In [ ]:
model = src.tfidf_svd50_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
model = src.tfidf_svd200_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
model = src.tfidf_svd300_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

In [ ]:
model = src.tfidf_svd500_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 